# Train Buddy’s models on CDC weekly reports

Choose **Runtime → Change runtime type → GPU**. Start with the **T4** and the default 8B model. The notebook fits one research adapter per run. A G4 with enough free memory can use the larger selections. The hardware check runs before any model weights download.

Each example represents a condition, reporting area and MMWR week. The labels describe reported case counts. Select **Smoke test** to exercise the training path or **Benchmark** for one pass over the prepared training split. GPU availability and runtime duration vary. The final cell downloads the adapter, metrics and split fingerprints. The notebook runs independently of Buddy and Cloudflare.

In [ ]:
MODEL = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B" # @param ["deepseek-ai/DeepSeek-R1-Distill-Llama-8B", "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B", "openai/gpt-oss-120b"]
PROFILE = "Smoke test" # @param ["Smoke test", "Benchmark"]
from pathlib import Path
import os, subprocess, sys, tempfile
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU in Runtime → Change runtime type, then rerun.")
free_gib = torch.cuda.mem_get_info()[0] / 2**30
print(torch.cuda.get_device_name(0), f"{free_gib:.1f} GiB free")

## Set up the training environment

The setup keeps Colab’s CUDA build of PyTorch and installs training packages in an isolated environment. Public model weights require substantial local storage. Use a fresh runtime if other models are still in memory.

In [ ]:
import urllib.request
work = Path("/content/healthbrowse-nndss")
work.mkdir(exist_ok=True)
revision = "0c96a0a1ded1ae84152b6028e282c892b12a34a4"
src = work / revision
src.mkdir(exist_ok=True)
for filename in ("prepare.py", "train.py"):
    url = f"https://raw.githubusercontent.com/abby2395/medical_chatbot/{revision}/healthbrowse/inference/nndss/{filename}"
    with urllib.request.urlopen(url, timeout=60) as resp:
        (src / filename).write_bytes(resp.read())
py = work / "venv/bin/python"
if not py.exists():
    subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", str(work / "venv")], check=True)
packages = ["transformers==4.57.6", "peft==0.17.1", "accelerate==1.10.1", "bitsandbytes==0.48.1"]
if MODEL == "openai/gpt-oss-120b":
    packages += ["kernels>=0.9,<1"]
subprocess.run([str(py), "-m", "pip", "install", "--quiet", *packages], check=True)
subprocess.run([str(py), "-c", "import sys; from train import require_gpu; require_gpu(sys.argv[1])", MODEL],
               cwd=src, check=True)

## Prepare the common benchmark

CDC permits anonymous access. If you want the notebook to use your Socrata token, add **SOCRATA_APP_TOKEN** under **Secrets** in Colab and grant this notebook access. A Cloudflare variable is separate. A second model run in the same runtime reuses the same prepared records for the selected profile.

In [ ]:
import json
from google.colab import userdata
data = work / ("data-" + revision[:8] + "-" + PROFILE.split()[0].lower())
if not (data / "benchmark.json").exists():
    env = os.environ.copy()
    try:
        env["SOCRATA_APP_TOKEN"] = userdata.get("SOCRATA_APP_TOKEN")
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        pass
    sizes = ["--train-size", "256", "--validation-size", "32", "--test-size", "100"] if PROFILE == "Smoke test" else []
    subprocess.run([str(py), str(src / "prepare.py"), "--out", str(data), *sizes], check=True, env=env)
    del env
benchmark = json.loads((data / "benchmark.json").read_text())
print(benchmark["years"], {k: v["rows"] for k, v in benchmark["splits"].items()})

## Train and evaluate

The smoke test uses 16 optimizer steps. The benchmark uses one step per training record. The trainer scores the base and the adapted model on the same held-out weeks. Large model downloads can take longer than the smoke test itself.

In [ ]:
run = Path(tempfile.mkdtemp(prefix=MODEL.split("/")[-1] + "-", dir=work)) / "result"
steps = ["--steps", "16"] if PROFILE == "Smoke test" else []
subprocess.run([str(py), str(src / "train.py"), "--model", MODEL, "--data", str(data), "--out", str(run), *steps], check=True)
metrics = json.loads((run / "metrics.json").read_text())
for name in ("base_test", "adapter_test"):
    print(name, {k: round(metrics[name][k], 4) for k in ("accuracy", "macro_f1")})

## Download this run

The ZIP includes the adapter weights, model revision, local evaluation metrics and the exact data splits. Keep it before disconnecting the runtime.

In [ ]:
import shutil
from google.colab import files
shutil.copytree(data, run / "benchmark", dirs_exist_ok=True)
archive = shutil.make_archive(str(run.parent / (MODEL.split("/")[-1] + "-nndss")), "zip", run)
files.download(archive)